In [0]:
import json

sample_records = [
    {
        "event_id": "E001",
        "event_timestamp": "2026-10-12T10:15:00Z",
        "vehicle": {
            "id": "V001",
            "make": "Toyota",
            "model": "Camry",
            "model_year": 2023,
            "price": 28000.0,
            "features": ["Hybrid", "Navigation"],
        },
        "dealer": {"id": "D01", "state": "CA"},
        "status": "AVAILABLE",
    },
    {
        "event_id": "E002",
        "event_timestamp": "2026-10-12T11:30:00Z",
        "vehicle": {
            "id": "V002",
            "make": "Toyota",
            "model": "Corolla",
            "model_year": 2022,
            "price": 22000.0,
            "features": ["Bluetooth"],
        },
        "dealer": {"id": "D01", "state": "CA"},
        "status": "AVAILABLE",
    },
    {
        "event_id": "E003",
        "event_timestamp": "2026-10-13T09:00:00Z",
        "vehicle": {
            "id": "V003",
            "make": "Honda",
            "model": "Civic",
            "model_year": 2024,
            "price": 26000.0,
            "features": ["Navigation", "Sunroof"],
        },
        "dealer": {"id": "D02", "state": "TX"},
        "status": "AVAILABLE",
    },
    {
        "event_id": "E004",
        "event_timestamp": "2026-10-13T12:45:00Z",
        "vehicle": {
            "id": "V004",
            "make": "Ford",
            "model": "Escape",
            "model_year": 2021,
            "price": 24000.0,
            "features": [],
        },
        "dealer": {"id": "D03", "state": "CA"},
        "status": "SOLD",
    },
]

raw_df = spark.createDataFrame(
    [(json.dumps(record),) for record in sample_records],
    "raw_json string",
)

display(raw_df)

In [0]:
from pyspark.sql import functions as F

json_schema = """
    event_id STRING,
    event_timestamp STRING,
    vehicle STRUCT<
        id: STRING,
        make: STRING,
        model: STRING,
        model_year: INT,
        price: DOUBLE,
        features: ARRAY<STRING>
    >,
    dealer STRUCT<
        id: STRING,
        state: STRING
    >,
    status STRING
"""

parsed_df = raw_df.withColumn(
    "parsed",
    F.from_json(F.col("raw_json"), json_schema),
)

display(parsed_df.select("parsed.*"))
parsed_df.select("parsed.*").printSchema()

In [0]:
flat_df = parsed_df.select(
    F.col("parsed.event_id").alias("event_id"),
    F.col("parsed.event_timestamp").alias("event_timestamp_raw"),
    F.col("parsed.vehicle.id").alias("vehicle_id"),
    F.upper(F.trim(F.col("parsed.vehicle.make"))).alias("make"),
    F.col("parsed.vehicle.model").alias("model"),
    F.col("parsed.vehicle.model_year").alias("model_year"),
    F.col("parsed.vehicle.price").alias("price"),
    F.col("parsed.vehicle.features").alias("features"),
    F.col("parsed.dealer.id").alias("dealer_id"),
    F.col("parsed.dealer.state").alias("dealer_state"),
    F.col("parsed.status").alias("status"),
)

display(flat_df)

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

enriched_df = (
    flat_df
    .withColumn(
        "event_ts",
        F.to_timestamp(
            "event_timestamp_raw",
            "yyyy-MM-dd'T'HH:mm:ssXXX",
        ),
    )
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.hour("event_ts"))
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("feature_count", F.size("features"))
)

display(enriched_df)

In [0]:
display(
    enriched_df.filter(
        F.col("event_timestamp_raw").isNotNull()
        & F.col("event_ts").isNull()
    )
)

In [0]:
from pyspark.sql import functions as F

enriched_df = enriched_df.withColumn(
    "price",
    F.when(
        F.col("vehicle_id") == "V001",
        F.lit(-100.0),
    ).otherwise(F.col("price")),
)

In [0]:
filtered_df = enriched_df.filter(
    (F.col("dealer_state")== "CA") 
    & (F.col("status")== "AVAILABLE")
)
display(filtered_df)


In [0]:
invalid_price_df = enriched_df.filter(
    F.col("price").isNull()
    | (F.col("price") <= 0)
)

invalid_count = invalid_price_df.count()

display(invalid_price_df)
print(f"Invalid price records: {invalid_count}")

In [0]:
if invalid_count > 0:
    raise ValueError(
        f"Data quality failed: {invalid_count} records "
        "have null or non-positive prices."
    )

print("Price quality check passed")

In [0]:
summary_df = (
    enriched_df
    .filter(F.col("status") == "AVAILABLE")
    .groupBy("make")
    .agg(
        F.count("*").alias("vehicle_count"),
        F.round(F.avg("price"), 2).alias("average_price"),
        F.sum("price").alias("total_price"),
        F.max("event_ts").alias("latest_event_ts"),
    )
    .orderBy("make")
)

display(summary_df)

In [0]:
features_df = enriched_df.select(
    "vehicle_id",
    "make",
    F.explode_outer("features").alias("feature"),
)

display(features_df)

In [0]:
enriched_df.createOrReplaceTempView("vehicle_events")

In [0]:
%sql
Select make, count(*) AS Vehicle_Count,
ROUND(AVG(price)) AS Average_Price,
SUM(price) AS Total_Price,
MAX(event_ts) AS Latest_Event_Timestamp FROM vehicle_events
where status = 'AVAILABLE'  group by make  
order by make